<a href="https://colab.research.google.com/github/shinichiwijaya/cv/blob/main/persona_cloner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kloning Kepribadian dari Chat WhatsApp
**Input:** export chat WhatsApp (.txt) → **Output:** System Prompt Kepribadian (Persona & Pola Respon)

Alur notebook ini:
1. **Parser** – baca export WA jadi tabel (waktu, pengirim, isi)
2. **Anonimisasi** – nomor, URL, email, dan nama diganti placeholder
3. **Fitur stilometrik** – gaya mengetik, *dihitung* dengan kode (bukan LLM)
4. **Ekstraksi kepribadian** – LLM (OpenRouter) menilai Big Five, gaya komunikasi, dan pola respon
5. **Rakit system prompt** + character card untuk SillyTavern

**Cara pakai:** isi sel *Konfigurasi*, lalu jalankan sel dari atas ke bawah.
Simpan API key OpenRouter di Colab: ikon kunci (Secrets) di sidebar kiri → nama `OPENROUTER_API_KEY`.

> Catatan etika: pakai hanya chat yang kamu punya hak/izin untuk dipakai. Minta persetujuan lawan bicara dan jangan kumpulkan chat mentah.

In [ ]:
import re, json, random, statistics, textwrap, os
from collections import Counter
import pandas as pd
import requests
print("Library siap. pandas", pd.__version__)

Library siap. pandas 2.2.3


## Konfigurasi

In [ ]:
CHAT_PATH = '/content/WhatsApp Chat with Faiz.txt'             # None = upload file lewat Colab. Atau isi path, mis. "/content/chat.txt"
TARGET_NAME = "shin"         # nama orang yang mau DIKLONING, persis seperti tertulis di chat
CHAR_NAME = "shin"           # nama karakter di SillyTavern
USER_ALIASES = ["Faiz", "faiz*"]  # panggilan lawan bicara → {{user}}. Tanda * di akhir = semua varian ejaan berawalan itu
CHAR_ALIASES = []            # panggilan lain untuk TARGET → diganti {{char}}
EXTRA_NAMES_TO_MASK = ["Eric", "Vera", "Winata"]   # nama pihak ketiga yang disensor → <NAMA>
DAYFIRST = None              # None = deteksi otomatis dari tanggal; True = hari/bulan; False = bulan/hari
PASTE_WORD_LIMIT = 60        # pesan > 60 kata dianggap tempelan (mis. jawaban ChatGPT) dan diabaikan dari analisis gaya
SESSION_GAP_HOURS = 6        # jeda >6 jam dianggap percakapan baru
MAX_PROMPT_CHARS = 60000     # batas panjang cuplikan chat yang dikirim ke LLM
N_EXAMPLES = 12              # jumlah contoh balasan asli untuk few-shot
SEED = 42
LLM_MODEL = "google/gemma-4-31b-it"   # model di OpenRouter

## Tahap 2a – Parser export WhatsApp
Mendukung format Android (`12/03/2025 14.05 - Nama: pesan`) dan iOS (`[12/03/2025, 14.05.33] Nama: pesan`).
Pesan multi-baris digabung, pesan sistem dan "media omitted" dibuang.

In [ ]:
TS = (r"(?P<date>\d{1,2}/\d{1,2}/\d{2,4}),?\s+"
      r"(?P<time>\d{1,2}[.:]\d{2}(?:[.:]\d{2})?(?:\s?[AaPp]\.?[Mm]\.?)?)")
RE_ANDROID = re.compile(rf"^{TS}\s+-\s+(?P<rest>.*)$")
RE_IOS = re.compile(rf"^\[{TS}\]\s+(?P<rest>.*)$")
RE_SENDER = re.compile(r"^([^:\n]{1,60}?):\s(.*)$", re.S)

NOISE = [r"media (omitted|tidak disertakan)",
         r"(gambar|video|audio|stiker|sticker|gif|dokumen|image|document|kontak|contact card) (omitted|tidak disertakan|dihilangkan)",
         r"pesan ini dihapus", r"this message was deleted", r"anda menghapus pesan ini",
         r"you deleted this message", r"^null$", r"menunggu pesan ini", r"waiting for this message"]

def is_noise(t):
    return len(t) < 80 and any(re.search(p, t.lower()) for p in NOISE)

MEDIA_RE = re.compile(r"media (omitted|tidak disertakan)|(gambar|video|audio|stiker|sticker|gif|dokumen|image|document) "
                      r"(omitted|tidak disertakan)", re.I)
MEDIA_COUNTS = {}

def is_media(t):
    return len(t) < 60 and bool(MEDIA_RE.search(t))

def is_pasted(t):
    """Teks tempelan panjang (mis. jawaban ChatGPT) – bukan gaya mengetik orangnya."""
    return len(t.split()) > PASTE_WORD_LIMIT or t.count("\n") >= 3 or "```" in t or t.count("**") >= 2

def parse_whatsapp(path, dayfirst=None):
    global MEDIA_COUNTS
    raw = open(path, encoding="utf-8", errors="ignore").read()
    for ch, rep in {"\u200e": "", "\u200f": "", "\ufeff": "", "\u202f": " ", "\u00a0": " "}.items():
        raw = raw.replace(ch, rep)
    rows, cur = [], None
    for line in raw.splitlines():
        m = RE_ANDROID.match(line) or RE_IOS.match(line)
        if m:
            if cur:
                rows.append(cur)
            cur = None
            ms = RE_SENDER.match(m["rest"])
            if not ms:          # pesan sistem (tanpa pengirim)
                continue
            tm = re.sub(r"([AaPp])\.([Mm])\.", r"\1\2", m["time"]).replace(".", ":")
            cur = {"date": m["date"], "time": tm, "sender": ms.group(1).strip(), "text": ms.group(2)}
        elif cur is not None:   # lanjutan pesan multi-baris
            cur["text"] += "\n" + line
    if cur:
        rows.append(cur)
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("Tidak ada pesan terbaca. Cek format export (buka file .txt dan lihat 3 baris pertama).")
    if dayfirst is None:
        a = df["date"].str.split("/").str[0].astype(int)
        b = df["date"].str.split("/").str[1].astype(int)
        if a.max() > 12:
            dayfirst = True
        elif b.max() > 12:
            dayfirst = False
        else:
            dayfirst = True
            print("Peringatan: urutan tanggal ambigu, diasumsikan hari/bulan. Set DAYFIRST=False jika salah.")
    print("Urutan tanggal:", "hari/bulan/tahun" if dayfirst else "bulan/hari/tahun")
    s = df["date"] + " " + df["time"]
    try:
        df["datetime"] = pd.to_datetime(s, dayfirst=dayfirst, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        df["datetime"] = pd.to_datetime(s, dayfirst=dayfirst, errors="coerce")
    bad = int(df["datetime"].isna().sum())
    if bad:
        print(f"Peringatan: {bad} baris dengan tanggal tidak terbaca dibuang.")
    df = df.dropna(subset=["datetime"])
    df["text"] = (df["text"].str.replace(r"\s*<(pesan ini diedit|this message was edited)>", "",
                                         regex=True, flags=re.I).str.strip())
    MEDIA_COUNTS = df[df["text"].apply(is_media)]["sender"].value_counts().to_dict()
    df = df[(df["text"] != "") & (~df["text"].apply(is_noise))]
    return df[["datetime", "sender", "text"]].reset_index(drop=True)

if CHAT_PATH is None:
    from google.colab import files
    _up = files.upload()
    CHAT_PATH = next(iter(_up))

df_raw = parse_whatsapp(CHAT_PATH, DAYFIRST)
print(f"{len(df_raw)} pesan | {df_raw.datetime.min():%d %b %Y} s/d {df_raw.datetime.max():%d %b %Y}")
print("\nPeserta di chat:")
print(df_raw.sender.value_counts().to_string())
print(f"\nMedia yang dikirim (foto/stiker/dll, isinya tidak dianalisis): {MEDIA_COUNTS}")
print(f"Pesan tempelan panjang (diabaikan dari analisis gaya): {int(df_raw.text.apply(is_pasted).sum())}")

def resolve_target(df, name):
    senders = list(df.sender.unique())
    for s in senders:
        if s.lower() == name.lower():
            return s
    hits = [s for s in senders if name.lower() in s.lower()]
    if len(hits) == 1:
        return hits[0]
    raise ValueError(f"TARGET_NAME '{name}' tidak ditemukan/ambigu. Pilih salah satu: {senders}")

TARGET_RAW = resolve_target(df_raw, TARGET_NAME)
print("\nTarget kloning:", TARGET_RAW)

Urutan tanggal: bulan/hari/tahun
6598 pesan | 28 Aug 2024 s/d 02 Oct 2026

Peserta di chat:
sender
Faiz    3672
shin    2926

Media yang dikirim (foto/stiker/dll, isinya tidak dianalisis): {'Faiz': 706, 'shin': 557}
Pesan tempelan panjang (diabaikan dari analisis gaya): 19

Target kloning: shin


## Tahap 2b – Anonimisasi
Pengirim diganti `TARGET` / `LAWAN`. Nama TARGET → `{{char}}`, nama lawan bicara → `{{user}}` (format SillyTavern), pihak ketiga → `<NAMA>`; nomor, URL, email juga disensor.

In [ ]:
RE_URL = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
RE_PHONE = re.compile(r"(?:\+?62|\b0)[\s-]?8\d{1,3}[\s-]?\d{3,4}[\s-]?\d{2,5}|\b\d{9,}\b")

def name_regex(names):
    """Regex nama, toleran huruf diperpanjang (shin → shinnn) dan huruf besar/kecil."""
    forms = set()
    for n in names:
        n = n.strip()
        if n.endswith("*") and len(n) >= 4:          # wildcard: "shin*" = semua kata berawalan "shin"
            forms.add(re.escape(n[:-1]) + r"\w*")
        elif len(n) >= 3 and re.search(r"[A-Za-z]", n):
            forms.add(re.escape(n) + re.escape(n[-1]) + "*(?:chan|kun|san|sama|nya)?")
    forms = sorted(forms, key=len, reverse=True)
    return re.compile(r"(?<!\w)(?:" + "|".join(forms) + r")(?!\w)", re.I) if forms else None

def name_variants(sender, aliases=()):
    out = set(aliases)
    if re.search(r"[A-Za-z]", sender):
        out.add(sender)
        out.update(w for w in re.split(r"\W+", sender) if len(w) >= 3 and not w.isdigit())
    return out

others = [s for s in df_raw.sender.unique() if s != TARGET_RAW]
NAME_RULES = [
    (name_regex(name_variants(TARGET_RAW, CHAR_ALIASES)), "{{char}}"),
    (name_regex(set().union(*[name_variants(s) for s in others]) | set(USER_ALIASES)), "{{user}}"),
    (name_regex(EXTRA_NAMES_TO_MASK), "<NAMA>"),
]

def anonymize_text(t):
    t = RE_URL.sub("<URL>", t)
    t = RE_EMAIL.sub("<EMAIL>", t)
    t = RE_PHONE.sub("<NOMOR>", t)
    for rx, repl in NAME_RULES:
        if rx:
            t = rx.sub(repl, t)
    return t

label = {TARGET_RAW: "TARGET"}
for i, s in enumerate(others, 1):
    label[s] = "LAWAN" if len(others) == 1 else f"LAWAN_{i}"

df = df_raw.copy()
df["sender"] = df["sender"].map(label)
df["text"] = df["text"].apply(anonymize_text)
df.to_csv("chat_anonim.csv", index=False)
print("Contoh hasil anonimisasi:")
print(df.head(8).to_string(index=False))
print("\nPesan 1 kata paling sering (cek: adakah nama yang lolos sensor? kalau ada, tambahkan ke USER_ALIASES / EXTRA_NAMES_TO_MASK):")
print(df[df.text.str.split().str.len() == 1].text.str.lower().value_counts().head(12).to_string())

Contoh hasil anonimisasi:
           datetime sender                                     text
2024-08-28 18:21:00  LAWAN                                 SHINICHI
2024-08-28 18:21:00 TARGET                                      iya
2024-08-28 18:21:00 TARGET                      sbb baru liat teams
2024-08-28 18:21:00  LAWAN                            lu kemana jir
2024-08-28 18:22:00 TARGET                       tadi tinggal makan
2024-08-28 18:22:00  LAWAN                                   owlaah
2024-08-28 18:22:00  LAWAN <NAMA> <NAMA> itu <NAMA> yg python bukan
2024-08-28 18:22:00 TARGET                            kayaknya beda

Pesan 1 kata paling sering (cek: adakah nama yang lolos sensor? kalau ada, tambahkan ke USER_ALIASES / EXTRA_NAMES_TO_MASK):
text
{{char}}    159
iya         134
absen        89
shinichi     85
wkwkw        81
okok         75
oke          51
{{user}}     38
eh           37
ok           35
<url>        32
wait         25


## Tahap 3 – Fitur stilometrik (dihitung, bukan ditebak LLM)

In [ ]:
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B50\u2B06\u2764\u203C\u2049]")
LAUGH_RE = re.compile(r"\b(?:(?:wk){2,}\w*|w[k]{3,}|(?:ha){2,}h?|(?:he){2,}h?|(?:hi){2,}h?|(?:xi){2,}x?|"
                      r"(?:kw){2,}\w*|awok\w*|lol|lmao|rofl)\b", re.I)
PLACEHOLDER = re.compile(r"<[A-Z_]+>|\{\{(?:user|char)\}\}")
SLANG = {"gak", "ga", "nggak", "ngga", "enggak", "engga", "nggk", "yg", "dgn", "dg", "utk", "aja", "aj", "banget",
         "bgt", "bngt", "sih", "deh", "dong", "lah", "kok", "yaudah", "yaudh", "udah", "udh", "gpp", "gapapa",
         "btw", "otw", "gitu", "gini", "gimana", "gmn", "kayak", "kek", "emang", "emg", "tuh", "nih", "yah", "yaa",
         "iya", "iyaa", "oke", "okeh", "sip", "mantap", "anjir", "anjay", "njir", "wah", "duh", "hmm", "bro",
         "bang", "kak", "cuy", "guys", "dah", "kan", "loh", "lho", "mah", "tau", "tp", "jd", "krn", "sm", "klo",
         "kalo", "kalau", "bsk", "besok", "lg", "lagi", "pls", "plis", "thx", "makasih", "tengkyu", "mksh",
         "gk", "dh", "ntar", "jir", "bejir", "pake", "tpi", "blm", "brp", "cmn", "sbb", "ngab", "woi", "wey"}
PRONOUNS = {"gue/lu": {"gue", "gw", "gua", "w", "lu", "lo", "loe", "elu", "elo"},
            "aku/kamu": {"aku", "ak", "km", "kamu", "kmu", "kau"},
            "saya/anda": {"saya", "anda"}}
STOP_ID = {"yang", "di", "ke", "dan", "itu", "ini", "yg", "dgn", "untuk", "dari", "ada", "sama", "juga", "ya",
           "nya", "aku", "kamu", "gak", "ga", "aja", "udah", "kan", "mau", "bisa", "apa", "kalo", "tapi", "sih"}
EN_WORDS = {"the", "is", "are", "and", "you", "i", "to", "of", "it", "that", "for", "in", "on", "my", "me", "we",
            "this", "with", "what", "so", "just", "can", "not", "but", "if", "have", "do", "how", "why", "yes",
            "no", "thanks", "please", "sorry", "love", "like", "good", "nice", "really", "actually", "anyway"}

def tok(t):
    return re.findall(r"[a-zA-Z']+", PLACEHOLDER.sub(" ", t).lower())

def r2(x, nd=2):
    return None if x is None else round(float(x), nd)

REDUP_RE = re.compile(r"\b([A-Za-z]{2,}?)\s?\1\b", re.I)

def compute_stats(d, media_count=0):
    d = d.copy().reset_index(drop=True)
    d["pasted"] = d.text.apply(is_pasted)
    n_pasted = int(((d.sender == "TARGET") & d.pasted).sum())
    d = d[~d.pasted].reset_index(drop=True)
    d["turn"] = (d.sender != d.sender.shift()).cumsum()
    d["session"] = (d.datetime.diff() > pd.Timedelta(hours=SESSION_GAP_HOURS)).cumsum()
    T = d[d.sender == "TARGET"]
    texts = T.text.tolist()
    n = len(texts)
    if n == 0:
        raise ValueError("Tidak ada pesan TARGET setelah filter.")
    pct = lambda c: r2(100 * c / n, 1)
    clean = [PLACEHOLDER.sub("", t).strip() for t in texts]
    wc = [len(t.split()) for t in texts]
    toks = [w for t in texts for w in tok(t)]
    ntok = max(len(toks), 1)

    emo = [EMOJI_RE.findall(t) for t in texts]
    emo_all = Counter(e for l in emo for e in l)
    laugh = [LAUGH_RE.findall(t) for t in texts]
    laugh_all = Counter(x.lower() for l in laugh for x in l)
    redup = [m for m in (REDUP_RE.search(LAUGH_RE.sub("", t)) for t in clean)
             if m and len(set(m.group(0).lower().replace(" ", ""))) > 1]
    redup_all = Counter(m.group(0).lower() for m in redup)
    alpha = [t for t in clean if re.search(r"[A-Za-z]", t)]
    first_alpha = [re.search(r"[A-Za-z]", t).group() for t in alpha]
    last_char = [t.rstrip()[-1:] for t in clean if t.strip()]
    tcount = Counter(toks)
    pron = {k: sum(tcount[w] for w in v) for k, v in PRONOUNS.items()}
    ptot = sum(pron.values())
    pron_pct = {k: (r2(100 * v / ptot, 1) if ptot else 0) for k, v in pron.items()}

    turns = d.groupby("turn").agg(sender=("sender", "first"), session=("session", "first"),
                                  start=("datetime", "first"), end=("datetime", "last"), n=("text", "size"))
    ps, pe, pss = turns.sender.shift(), turns.end.shift(), turns.session.shift()
    mask = (turns.sender == "TARGET") & ps.notna() & (ps != "TARGET") & (turns.session == pss)
    lat = ((turns.start[mask] - pe[mask]).dt.total_seconds() / 60).tolist()
    first_in_session = d.groupby("session").sender.first()
    hours = Counter(T.datetime.dt.hour)

    return {
        "n_pesan_target": n, "n_pesan_total": len(d), "porsi_target_persen": r2(100 * n / len(d), 1),
        "pesan_tempelan_diabaikan": n_pasted,
        "rentang_tanggal": [str(d.datetime.min().date()), str(d.datetime.max().date())],
        "kata_per_pesan_rata2": r2(statistics.mean(wc)), "kata_per_pesan_median": statistics.median(wc),
        "kata_per_pesan_p90": r2(sorted(wc)[int(0.9 * (n - 1))]), "karakter_per_pesan_rata2": r2(statistics.mean(len(t) for t in texts), 1),
        "pesan_pendek_le3kata_persen": pct(sum(w <= 3 for w in wc)),
        "bubble_per_giliran_rata2": r2(turns[turns.sender == "TARGET"].n.mean()),
        "media_per_100_pesan_teks": r2(100 * media_count / n, 1),
        "emoji_per_pesan": r2(sum(len(l) for l in emo) / n, 3), "pesan_beremoji_persen": pct(sum(bool(l) for l in emo)),
        "emoji_teratas": [[e, c] for e, c in emo_all.most_common(5)],
        "pesan_tawa_persen": pct(sum(bool(l) for l in laugh)), "tawa_teratas": [[w, c] for w, c in laugh_all.most_common(4)],
        "pesan_kata_diulang_persen": pct(len(redup)), "kata_diulang_contoh": [w for w, _ in redup_all.most_common(4)],
        "pesan_huruf_kecil_semua_persen": pct(sum(t == t.lower() for t in alpha)),
        "awal_huruf_kecil_persen": r2(100 * sum(c.islower() for c in first_alpha) / max(len(first_alpha), 1), 1),
        "kata_kapital_per_100pesan": r2(100 * sum(len([w for w in re.findall(r"[A-Za-z]{3,}", t) if w.isupper()]) for t in clean) / n, 1),
        "pesan_huruf_diperpanjang_persen": pct(sum(bool(re.search(r"([a-zA-Z])\1{2,}", t)) for t in clean)),
        "akhir_titik_persen": r2(100 * sum(c == "." for c in last_char) / max(len(last_char), 1), 1),
        "akhir_tanya_persen": r2(100 * sum(c == "?" for c in last_char) / max(len(last_char), 1), 1),
        "akhir_seru_persen": r2(100 * sum(c == "!" for c in last_char) / max(len(last_char), 1), 1),
        "tanpa_tanda_baca_akhir_persen": r2(100 * sum(not re.match(r"[.?!]", c) for c in last_char) / max(len(last_char), 1), 1),
        "pesan_elipsis_persen": pct(sum("..." in t for t in clean)),
        "pesan_mengandung_tanya_persen": pct(sum("?" in t for t in clean)),
        "panggil_lawan_persen": pct(sum("{{user}}" in t for t in texts)),
        "kata_teratas": [[w, c] for w, c in tcount.most_common(60) if w not in STOP_ID and len(w) > 1][:15],
        "slang_teratas": [[w, c] for w, c in Counter({w: tcount[w] for w in SLANG if tcount[w]}).most_common(12)],
        "kata_ganti": pron, "kata_ganti_persen": pron_pct,
        "kata_ganti_dominan": (max(pron, key=pron.get) if ptot else "tidak menonjol"),
        "kata_inggris_persen": r2(100 * sum(tcount[w] for w in EN_WORDS) / ntok, 1),
        "pembuka_teratas": [[w, c] for w, c in Counter(next(iter(tok(t)), "") for t in clean).most_common(8) if w],
        "jeda_balas_median_menit": r2(statistics.median(lat)) if lat else None,
        "jeda_balas_p75_menit": r2(sorted(lat)[int(0.75 * (len(lat) - 1))]) if lat else None,
        "balas_kurang_1menit_persen": r2(100 * sum(x < 1 for x in lat) / len(lat), 1) if lat else None,
        "inisiasi_percakapan_persen": r2(100 * (first_in_session == "TARGET").mean(), 1),
        "jam_aktif_teratas": [h for h, _ in hours.most_common(3)],
        "pesan_larut_malam_0_4_persen": pct(sum(v for h, v in hours.items() if h < 5)),
    }

stats = compute_stats(df, MEDIA_COUNTS.get(TARGET_RAW, 0))
json.dump(stats, open("stylometry.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(stats, ensure_ascii=False, indent=1))

{
 "n_pesan_target": 2922,
 "n_pesan_total": 6579,
 "porsi_target_persen": 44.4,
 "pesan_tempelan_diabaikan": 4,
 "rentang_tanggal": [
  "2024-08-28",
  "2026-10-02"
 ],
 "kata_per_pesan_rata2": 3.67,
 "kata_per_pesan_median": 3.0,
 "kata_per_pesan_p90": 7.0,
 "karakter_per_pesan_rata2": 18.8,
 "pesan_pendek_le3kata_persen": 58.2,
 "bubble_per_giliran_rata2": 1.74,
 "media_per_100_pesan_teks": 19.1,
 "emoji_per_pesan": 0.002,
 "pesan_beremoji_persen": 0.2,
 "emoji_teratas": [
  [
   "💀",
   2
  ],
  [
   "🙌",
   1
  ],
  [
   "😳",
   1
  ],
  [
   "😭",
   1
  ],
  [
   "🗿",
   1
  ]
 ],
 "pesan_tawa_persen": 9.4,
 "tawa_teratas": [
  [
   "wkwkw",
   258
  ],
  [
   "wkwkww",
   7
  ],
  [
   "wkwkwkw",
   6
  ],
  [
   "wkwkwk",
   2
  ]
 ],
 "pesan_kata_diulang_persen": 8.4,
 "kata_diulang_contoh": [
  "ok ok",
  "okok",
  "oke oke",
  "sama sama"
 ],
 "pesan_huruf_kecil_semua_persen": 94.5,
 "awal_huruf_kecil_persen": 97.3,
 "kata_kapital_per_100pesan": 3.6,
 "pesan_huruf_diperpanja

## Draft contoh perhitungan (untuk deliverable opsional #5)
Hitung manual 5 pesan pertama TARGET langkah demi langkah. Salin hasilnya ke Word/PDF.

In [ ]:
def show_manual_calc(d, k=5):
    T = d[(d.sender == "TARGET") & (d.text.str.split().str.len().between(3, 15))].head(k)
    rows = []
    for t in T.text:
        c = PLACEHOLDER.sub("", t).strip()
        rows.append({"pesan": c[:60], "kata": len(c.split()), "karakter": len(c),
                     "emoji": len(EMOJI_RE.findall(c)), "tawa": len(LAUGH_RE.findall(c)),
                     "huruf_kecil_semua": int(c == c.lower())})
    tab = pd.DataFrame(rows)
    print(tab.to_string(index=False))
    kata = tab["kata"].tolist()
    print(f"\nRata-rata kata/pesan = ({' + '.join(map(str, kata))}) / {len(kata)} = {sum(kata)/len(kata):.2f}")
    print(f"Median kata/pesan    = {statistics.median(kata)}")
    print(f"Emoji per pesan      = {tab.emoji.sum()} / {len(tab)} = {tab.emoji.sum()/len(tab):.2f}")
    print(f"Pesan huruf kecil    = {tab.huruf_kecil_semua.sum()} / {len(tab)} = {100*tab.huruf_kecil_semua.sum()/len(tab):.0f}%")

show_manual_calc(df)

                  pesan  kata  karakter  emoji  tawa  huruf_kecil_semua
    sbb baru liat teams     4        19      0     0                  1
     tadi tinggal makan     3        18      0     0                  1
python online seinget w     4        23      0     0                  1
  tadi w diluar soalnya     4        21      0     0                  1
            adu ilang 1     3        11      0     0                  1

Rata-rata kata/pesan = (4 + 3 + 4 + 4 + 3) / 5 = 3.60
Median kata/pesan    = 4
Emoji per pesan      = 0 / 5 = 0.00
Pesan huruf kecil    = 5 / 5 = 100%


## Tahap 4 – Ekstraksi kepribadian dengan LLM (OpenRouter)
Dikirim: cuplikan percakapan (diambil acak dari seluruh rentang waktu) + angka stilometrik.
LLM menghasilkan profil terstruktur (JSON): Big Five, dimensi gaya, ringkasan, dan pola respon.
Skor Big Five di sini adalah **inferensi dari teks**, bukan hasil tes psikologi.

In [ ]:
import os
import re
import json
import random
import requests
import pandas as pd

# Default model on Fireworks AI
LLM_MODEL = "accounts/fireworks/models/deepseek-v4p1-flash"

def get_key():
    try:
        from google.colab import userdata
        return userdata.get("FIREWORKS_API_KEY")
    except Exception:
        import getpass
        return os.environ.get("FIREWORKS_API_KEY") or getpass.getpass("Fireworks API key: ")

_KEY = None
def call_llm(messages, temperature=0.3, max_tokens=4000, model=None, **kwargs):
    global _KEY
    _KEY = _KEY or get_key()

    payload = {
        "model": model or LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "response_format": {"type": "json_object"}  # Forces Fireworks models to emit valid JSON
    }
    payload.update(kwargs)

    r = requests.post(
        "https://api.fireworks.ai/inference/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {_KEY}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=240
    )
    if r.status_code != 200:
        raise RuntimeError(f"Fireworks API error {r.status_code}: {r.text[:300]}")
    return r.json()["choices"][0]["message"].get("content") or ""

def sample_dialogue(d, max_chars, seed=SEED):
    d = d.copy().reset_index(drop=True)
    d["session"] = (d.datetime.diff() > pd.Timedelta(hours=SESSION_GAP_HOURS)).cumsum()
    sess = [g for _, g in d.groupby("session") if (g.sender == "TARGET").sum() >= 2]
    random.Random(seed).shuffle(sess)
    chosen, total = [], 0
    for g in sess:
        block = "\n".join(f"{s}: " + ("[teks tempelan panjang dilewati]" if is_pasted(t) else t[:300])
                          for s, t in zip(g.sender, g.text))
        if total + len(block) > max_chars:
            continue
        chosen.append((g.datetime.iloc[0], block))
        total += len(block)
    chosen.sort(key=lambda x: x[0])
    return "\n\n--- percakapan baru ---\n".join(b for _, b in chosen), len(chosen)

SCHEMA = """{
 "big_five": {
  "openness": {"skor": 0, "bukti": ["kutipan pendek"]},
  "conscientiousness": {"skor": 0, "bukti": []},
  "extraversion": {"skor": 0, "bukti": []},
  "agreeableness": {"skor": 0, "bukti": []},
  "neuroticism": {"skor": 0, "bukti": []}
 },
 "dimensi_gaya": {
  "formalitas": {"skor": 0, "bukti": []}, "kehangatan": {"skor": 0, "bukti": []},
  "humor": {"skor": 0, "bukti": []}, "sarkasme": {"skor": 0, "bukti": []},
  "ekspresivitas_emosi": {"skor": 0, "bukti": []}, "ketegasan": {"skor": 0, "bukti": []},
  "rasa_ingin_tahu_bertanya": {"skor": 0, "bukti": []}
 },
 "ringkasan_kepribadian": "3-5 kalimat, sudut pandang orang ketiga",
 "cara_bicara": "3-5 kalimat tentang gaya bicara khas (pilihan kata, nada, kebiasaan)",
 "pola_respon": [{"situasi": "...", "respon_khas": "...", "contoh_kutipan": "..."}],
 "topik_favorit": ["..."],
 "hal_yang_dihindari": ["..."],
 "fakta_personal_aman": ["fakta umum non-sensitif yang jelas disebut TARGET sendiri"],
 "tingkat_keyakinan": "rendah|sedang|tinggi",
 "catatan_keterbatasan": "..."
}"""

def build_extraction_messages(dialogue, stats):
    system = ("Kamu analis linguistik forensik dan psikologi kepribadian. Kamu menilai kepribadian seseorang "
              "HANYA dari bukti di chat. Jangan mengarang. Keluaran harus JSON valid saja, tanpa teks lain.")
    user = f"""Di bawah ini cuplikan chat WhatsApp. Pengirim 'TARGET' adalah orang yang dianalisis; 'LAWAN' adalah lawan bicaranya.
Nama sudah disensor: nama TARGET jadi {{{{char}}}}, nama lawan bicara jadi {{{{user}}}}, pihak ketiga jadi <NAMA>; nomor jadi <NOMOR>.
Teks tempelan panjang (mis. jawaban AI) sudah dilewati dan bukan gaya TARGET.

ATURAN:
- Skor 0-100 (50 = rata-rata populasi umum). Nilai ekstrem hanya jika bukti kuat.
- 'bukti' = kutipan asli TARGET, maksimal 8 kata per kutipan, 1-3 kutipan per item.
- 'pola_respon': 6-8 item tentang bagaimana TARGET biasa menanggapi situasi (mis. ditanya hal sepele, diajak bertemu,
  dikritik, dicurhati, diajak bercanda, tidak setuju, dipuji, menolak ajakan). Hanya yang terbukti di chat.
- Jika tidak ada bukti untuk suatu hal, isi null atau daftar kosong. Jangan menebak fakta pribadi.
- 'fakta_personal_aman' tidak boleh berisi data sensitif (kesehatan, agama, politik, alamat, keuangan).
- Isi 'tingkat_keyakinan' dan 'catatan_keterbatasan' dengan jujur.

ANGKA STILOMETRIK TARGET (hasil hitung kode, sudah akurat):
{json.dumps(stats, ensure_ascii=False)}

FORMAT KELUARAN (JSON saja):
{SCHEMA}

CUPLIKAN CHAT:
{dialogue}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def parse_json_loose(text):
    t = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
    t = re.sub(r"\s*```$", "", t, flags=re.I)

    s, e = t.find("{"), t.rfind("}")
    if s == -1 or e == -1:
        raise ValueError("No JSON object structure found in response.")
    json_str = t[s:e + 1]
    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)
    return json.loads(json_str)

def extract_profile(dialogue, stats, tries=3):
    msgs = build_extraction_messages(dialogue, stats)
    for i in range(tries):
        out = call_llm(msgs)
        try:
            return parse_json_loose(out)
        except Exception as ex:
            print(f"Percobaan {i+1}: JSON tidak valid ({ex}). Mencoba lagi...")
            msgs += [{"role": "assistant", "content": out},
                     {"role": "user", "content": "Keluaranmu bukan JSON valid. Kirim ulang HANYA JSON valid sesuai format schema yang diminta, pastikan tidak ada koma menggantung (trailing commas) atau kurung kurawal yang kurang."}]
    raise RuntimeError("Gagal mendapat JSON valid dari model.")

def clean_profile(p):
    for grp in ("big_five", "dimensi_gaya"):
        g = p.get(grp) or {}
        for k, v in list(g.items()):
            v = v if isinstance(v, dict) else {"skor": v, "bukti": []}
            try:
                v["skor"] = max(0, min(100, int(round(float(v.get("skor", 50))))))
            except Exception:
                v["skor"] = 50
            v.setdefault("buktfw_7rPb7WyqKGEpkQmTzKT6cmi", [])
            g[k] = v
        p[grp] = g
    return p

dialogue, n_sessions = sample_dialogue(df, MAX_PROMPT_CHARS)
print(f"Cuplikan: {n_sessions} percakapan, {len(dialogue)} karakter (~{len(dialogue)//4} token)")
profile = clean_profile(extract_profile(dialogue, stats))
json.dump(profile, open("persona_profile.json", "w"), ensure_ascii=False, indent=2)

print("\nBig Five:")
for k, v in profile["big_five"].items():
    print(f"  {k:<18} {v['skor']:>3}   {v['bukti'][:2]}")
print("\nRingkasan:", profile.get("ringkasan_kepribadian"))
print("Keyakinan:", profile.get("tingkat_keyakinan"), "|", profile.get("catatan_keterbatasan"))

Cuplikan: 74 percakapan, 61879 karakter (~15469 token)
Fireworks API key: ··········

Big Five:
  openness            72   ['pengen pake yang di neo4j itu', 'dulu tertarik nyoba htb ctf']
  conscientiousness   48   ['w lupa', 'last minute astaga']
  extraversion        50   ['wkwkw', 'ok ok']
  agreeableness       78   ['sama samaaa', 'kalo lu mau oke sih']
  neuroticism         42   ['ketar ketir', 'w jadi agak takut dosennya']

Ringkasan: {{char}} adalah mahasiswa IT yang santai, ramah, dan sangat kooperatif dalam kerja kelompok. Ia cenderung mengutamakan keharmonisan dan kesediaan membantu teman, sering kali mengorbankan waktu pribadinya untuk menutup tugas orang lain. Gaya komunikasinya singkat, informal, dan penuh humor receh, namun ia cukup penasaran pada topik teknis seperti keamanan siber, machine learning, dan pengembangan web. Ia agak kurang terorganisir dan sering mengerjakan tugas secara mepet, tetapi tetap berusaha menyelesaikan tanggung jawabnya. Secara emosional ia stabi

In [ ]:
def get_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except Exception:
        import getpass
        return os.environ.get("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ")

_KEY = None
def call_llm(messages, temperature=0.3, max_tokens=4000, model=None, **kwargs):
    global _KEY
    _KEY = _KEY or get_key()

    # Masukkan parameter dasar
    payload = {
        "model": model or LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    # Masukkan parameter tambahan (seperti top_p, presence_penalty, dll.) jika diberikan
    payload.update(kwargs)

    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {_KEY}", "Content-Type": "application/json",
                               "HTTP-Referer": "https://colab.research.google.com", "X-Title": "UTS Persona Cloning"},
                      json=payload, timeout=240)
    if r.status_code != 200:
        raise RuntimeError(f"OpenRouter error {r.status_code}: {r.text[:300]}")
    return r.json()["choices"][0]["message"].get("content") or ""

def sample_dialogue(d, max_chars, seed=SEED):
    d = d.copy().reset_index(drop=True)
    d["session"] = (d.datetime.diff() > pd.Timedelta(hours=SESSION_GAP_HOURS)).cumsum()
    sess = [g for _, g in d.groupby("session") if (g.sender == "TARGET").sum() >= 2]
    random.Random(seed).shuffle(sess)
    chosen, total = [], 0
    for g in sess:
        block = "\n".join(f"{s}: " + ("[teks tempelan panjang dilewati]" if is_pasted(t) else t[:300])
                          for s, t in zip(g.sender, g.text))
        if total + len(block) > max_chars:
            continue
        chosen.append((g.datetime.iloc[0], block))
        total += len(block)
    chosen.sort(key=lambda x: x[0])
    return "\n\n--- percakapan baru ---\n".join(b for _, b in chosen), len(chosen)

SCHEMA = """{
 "big_five": {
  "openness": {"skor": 0, "bukti": ["kutipan pendek"]},
  "conscientiousness": {"skor": 0, "bukti": []},
  "extraversion": {"skor": 0, "bukti": []},
  "agreeableness": {"skor": 0, "bukti": []},
  "neuroticism": {"skor": 0, "bukti": []}
 },
 "dimensi_gaya": {
  "formalitas": {"skor": 0, "bukti": []}, "kehangatan": {"skor": 0, "bukti": []},
  "humor": {"skor": 0, "bukti": []}, "sarkasme": {"skor": 0, "bukti": []},
  "ekspresivitas_emosi": {"skor": 0, "bukti": []}, "ketegasan": {"skor": 0, "bukti": []},
  "rasa_ingin_tahu_bertanya": {"skor": 0, "bukti": []}
 },
 "ringkasan_kepribadian": "3-5 kalimat, sudut pandang orang ketiga",
 "cara_bicara": "3-5 kalimat tentang gaya bicara khas (pilihan kata, nada, kebiasaan)",
 "pola_respon": [{"situasi": "...", "respon_khas": "...", "contoh_kutipan": "..."}],
 "topik_favorit": ["..."],
 "hal_yang_dihindari": ["..."],
 "fakta_personal_aman": ["fakta umum non-sensitif yang jelas disebut TARGET sendiri"],
 "tingkat_keyakinan": "rendah|sedang|tinggi",
 "catatan_keterbatasan": "..."
}"""

def build_extraction_messages(dialogue, stats):
    system = ("Kamu analis linguistik forensik dan psikologi kepribadian. Kamu menilai kepribadian seseorang "
              "HANYA dari bukti di chat. Jangan mengarang. Keluaran harus JSON valid saja, tanpa teks lain.")
    user = f"""Di bawah ini cuplikan chat WhatsApp. Pengirim 'TARGET' adalah orang yang dianalisis; 'LAWAN' adalah lawan bicaranya.
Nama sudah disensor: nama TARGET jadi {{{{char}}}}, nama lawan bicara jadi {{{{user}}}}, pihak ketiga jadi <NAMA>; nomor jadi <NOMOR>.
Teks tempelan panjang (mis. jawaban AI) sudah dilewati dan bukan gaya TARGET.

ATURAN:
- Skor 0-100 (50 = rata-rata populasi umum). Nilai ekstrem hanya jika bukti kuat.
- 'bukti' = kutipan asli TARGET, maksimal 8 kata per kutipan, 1-3 kutipan per item.
- 'pola_respon': 6-8 item tentang bagaimana TARGET biasa menanggapi situasi (mis. ditanya hal sepele, diajak bertemu,
  dikritik, dicurhati, diajak bercanda, tidak setuju, dipuji, menolak ajakan). Hanya yang terbukti di chat.
- Jika tidak ada bukti untuk suatu hal, isi null atau daftar kosong. Jangan menebak fakta pribadi.
- 'fakta_personal_aman' tidak boleh berisi data sensitif (kesehatan, agama, politik, alamat, keuangan).
- Isi 'tingkat_keyakinan' dan 'catatan_keterbatasan' dengan jujur.

ANGKA STILOMETRIK TARGET (hasil hitung kode, sudah akurat):
{json.dumps(stats, ensure_ascii=False)}

FORMAT KELUARAN (JSON saja):
{SCHEMA}

CUPLIKAN CHAT:
{dialogue}"""
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def parse_json_loose(text):
    # Remove markdown code blocks if present
    t = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.I)
    t = re.sub(r"\s*```$", "", t, flags=re.I)

    s, e = t.find("{"), t.rfind("}")
    if s == -1 or e == -1:
        raise ValueError("No JSON object structure found in response.")
    json_str = t[s:e + 1]

    # Clean up trailing commas before closing braces/brackets
    json_str = re.sub(r",\s*([}\]])", r"\1", json_str)

    return json.loads(json_str)

def extract_profile(dialogue, stats, tries=3):
    msgs = build_extraction_messages(dialogue, stats)
    for i in range(tries):
        out = call_llm(msgs)
        try:
            return parse_json_loose(out)
        except Exception as ex:
            print(f"Percobaan {i+1}: JSON tidak valid ({ex}). Mencoba lagi...")
            msgs += [{"role": "assistant", "content": out},
                     {"role": "user", "content": "Keluaranmu bukan JSON valid. Kirim ulang HANYA JSON valid sesuai format schema yang diminta, pastikan tidak ada koma menggantung (trailing commas) atau kurung kurawal yang kurang."}]
    raise RuntimeError("Gagal mendapat JSON valid dari model.")

def clean_profile(p):
    for grp in ("big_five", "dimensi_gaya"):
        g = p.get(grp) or {}
        for k, v in list(g.items()):
            v = v if isinstance(v, dict) else {"skor": v, "bukti": []}
            try:
                v["skor"] = max(0, min(100, int(round(float(v.get("skor", 50))))))
            except Exception:
                v["skor"] = 50
            v.setdefault("bukti", [])
            g[k] = v
        p[grp] = g
    return p

dialogue, n_sessions = sample_dialogue(df, MAX_PROMPT_CHARS)
print(f"Cuplikan: {n_sessions} percakapan, {len(dialogue)} karakter (~{len(dialogue)//4} token)")
profile = clean_profile(extract_profile(dialogue, stats))
json.dump(profile, open("persona_profile.json", "w"), ensure_ascii=False, indent=2)

print("\nBig Five:")
for k, v in profile["big_five"].items():
    print(f"  {k:<18} {v['skor']:>3}   {v['bukti'][:2]}")
print("\nRingkasan:", profile.get("ringkasan_kepribadian"))
print("Keyakinan:", profile.get("tingkat_keyakinan"), "|", profile.get("catatan_keterbatasan"))

Cuplikan: 74 percakapan, 61879 karakter (~15469 token)
Fireworks API key: ··········


RuntimeError: Fireworks API error 404: {"error":{"message":"Model not found, inaccessible, and/or not deployed","param":"model","code":"NOT_FOUND","type":"error"},"request_id":"chatcmpl-615e6462456a43b1925bf1f86eead4c9"}

### (Opsional) Validasi dengan kuesioner
Kalau temanmu/kamu mengisi BFI-2 versi Indonesia, masukkan rata-rata tiap trait (skala 1-5) untuk mengukur akurasi tebakan LLM.
Hasilnya bisa jadi bagian evaluasi di paper. Biarkan `None` kalau belum ada.

In [ ]:
SELF_REPORT_1_5 = {"openness": None, "conscientiousness": None, "extraversion": None,
                   "agreeableness": None, "neuroticism": None}
errs = []
for k, v in SELF_REPORT_1_5.items():
    if v is not None:
        gt = (v - 1) / 4 * 100
        pred = profile["big_five"][k]["skor"]
        errs.append(abs(gt - pred))
        print(f"{k:<18} kuesioner={gt:5.1f}  prediksi={pred:3d}  selisih={abs(gt-pred):5.1f}")
print(f"MAE = {statistics.mean(errs):.1f} poin (skala 0-100)" if errs else "Belum ada data kuesioner.")

Belum ada data kuesioner.


## Tahap 5 – Rakit System Prompt + Character Card SillyTavern
Skor Big Five diterjemahkan jadi **perilaku di chat** (bukan sekadar angka), dan gaya mengetik diambil dari angka stilometrik.
Contoh balasan asli (few-shot) dipilih otomatis dari chat.

In [ ]:
def pick_examples(d, n=N_EXAMPLES, seed=SEED):
    d = d.copy().reset_index(drop=True)
    d["turn"] = (d.sender != d.sender.shift()).cumsum()
    d["session"] = (d.datetime.diff() > pd.Timedelta(hours=SESSION_GAP_HOURS)).cumsum()
    turns = []
    for (_, s, sess), g in d.groupby(["turn", "sender", "session"], sort=False):
        turns.append((s, sess, list(g.text)))
    pairs = []
    for i in range(1, len(turns)):
        ps, psess, ptxt = turns[i - 1]
        s, sess, txt = turns[i]
        if s == "TARGET" and ps != "TARGET" and sess == psess:
            ctx, rep = "\n".join(ptxt[-2:])[:300], txt[:5]
            if (0 < len(" ".join(rep)) <= 400 and not re.search(r"<[A-Z_]+>", ctx + " ".join(rep))
                    and not any(is_pasted(x) for x in ptxt[-2:] + txt)):
                pairs.append({"konteks": ctx, "balasan": rep})
    bucket = lambda p: 0 if len(" ".join(p["balasan"]).split()) <= 3 else (1 if len(" ".join(p["balasan"]).split()) <= 12 else 2)
    rnd = random.Random(seed)
    groups = [[p for p in pairs if bucket(p) == b] for b in range(3)]
    for g in groups:
        rnd.shuffle(g)
    out, k = [], 0
    while len(out) < n and any(groups):
        if groups[k % 3]:
            out.append(groups[k % 3].pop())
        k += 1
        if k > 10 * n:
            break
    return out

examples = pick_examples(df)
json.dump(examples, open("examples.json", "w"), ensure_ascii=False, indent=2)
print(f"{len(examples)} contoh balasan terpilih. Contoh pertama:", examples[0] if examples else None)

12 contoh balasan terpilih. Contoh pertama: {'konteks': 'YG SECTION\nAWKWAKWAKWAKA', 'balasan': ['AISH WKKWW', 'KERJAIN']}


In [ ]:
TRAIT_DESC = {
    "openness": ("cenderung praktis dan konvensional; jarang mengulik topik abstrak atau hal baru",
                 "penasaran, suka ide dan topik baru; sering melempar pertanyaan atau gagasan di luar topik utama"),
    "conscientiousness": ("spontan dan santai; kadang menunda atau lupa, balasan tidak terlalu terstruktur",
                          "teratur dan bisa diandalkan; ingat detail, mengonfirmasi rencana, menepati janji"),
    "extraversion": ("pendiam dan hemat kata; menunggu diajak bicara, jarang memulai obrolan",
                     "ramai dan energik; sering memulai obrolan, banyak bicara, mudah akrab"),
    "agreeableness": ("blak-blakan; tidak sungkan menolak, mengkritik, atau berdebat",
                      "hangat dan suportif; menghindari konflik, mudah mengalah, memberi empati"),
    "neuroticism": ("tenang dan stabil; jarang mengeluh atau panik",
                    "mudah cemas; sering mengeluh, overthinking, ekspresif soal stres"),
}
TRAIT_ID = {"openness": "Keterbukaan", "conscientiousness": "Kehati-hatian", "extraversion": "Ekstraversi",
            "agreeableness": "Keramahan", "neuroticism": "Neurotisisme"}

def lvl(s):
    return "rendah" if s <= 35 else ("tinggi" if s >= 65 else "sedang")

def trait_line(k, s, bukti=None):
    lo, hi = TRAIT_DESC[k]
    desc = lo if s <= 35 else (hi if s >= 65 else "tidak menonjol; ikuti konteks percakapan")
    line = f"- {TRAIT_ID[k]} ({lvl(s)}): {desc}"
    if bukti and len(bukti) > 0:
        line += f" (Contoh ekspresi asli: {', '.join(repr(x) for x in bukti)})"
    return line + "."

def style_rules(st):
    L = []

    # ============================================================
    # PANJANG & STRUKTUR PESAN
    # ============================================================
    if st["kata_per_pesan_rata2"] <= 3:
        L.append("Pesan biasanya sangat singkat dan langsung ke inti. Sering kali satu bubble hanya berisi satu sampai beberapa kata.")
    elif st["kata_per_pesan_rata2"] <= 6:
        L.append("Pesan umumnya singkat dan langsung ke inti, tetapi dapat menjadi lebih panjang ketika konteks membutuhkan penjelasan.")
    else:
        L.append("Pesan cenderung ringkas, tetapi panjangnya dapat berubah mengikuti konteks percakapan.")

    if st["kata_per_pesan_p90"] <= 6:
        L.append("Sebagian besar balasan tetap pendek; jangan membuat setiap respons menjadi paragraf panjang kecuali memang sedang menjelaskan sesuatu.")

    # ============================================================
    # MULTI-BUBBLE
    # ============================================================
    if st["bubble_per_giliran_rata2"] >= 2:
        L.append("Sering mengirim beberapa bubble pendek secara berurutan daripada menggabungkan semuanya menjadi satu pesan panjang. Jika ada beberapa pikiran yang muncul berturut-turut, pisahkan menjadi beberapa baris.")

    # ============================================================
    # HURUF KECIL & TANDA BACA
    # ============================================================
    if st["pesan_huruf_kecil_semua_persen"] >= 70:
        L.append("Biasanya menggunakan huruf kecil dalam percakapan sehari-hari. Kapitalisasi tidak perlu dibuat formal.")
    elif st["pesan_huruf_kecil_semua_persen"] >= 40:
        L.append("Sering menggunakan huruf kecil dalam percakapan santai.")

    if st["tanpa_tanda_baca_akhir_persen"] >= 70:
        L.append("Sering tidak menggunakan tanda baca di akhir pesan. Balasan dapat terasa seperti chat spontan daripada tulisan formal.")
    elif st["tanpa_tanda_baca_akhir_persen"] >= 40:
        L.append("Tidak selalu menggunakan tanda baca di akhir pesan, terutama dalam percakapan santai.")

    # ============================================================
    # HURUF KAPITAL UNTUK EKSPRESI
    # ============================================================
    if st["pesan_huruf_diperpanjang_persen"] >= 3:
        L.append("Kadang memanjangkan huruf untuk mengekspresikan emosi atau penekanan, misalnya 'iyaaa', 'gakkk', atau bentuk serupa.")

    # ============================================================
    # EMOJI
    # ============================================================
    if st["emoji_per_pesan"] < 0.05:
        rare = [e for e, c in st["emoji_teratas"][:3] if c >= 5]
        if rare:
            L.append(f"Emoji jarang digunakan dan biasanya hanya muncul sesekali sebagai ekspresi. Emoji yang pernah digunakan antara lain: {' '.join(rare)}.")
        else:
            L.append("Emoji sangat jarang digunakan dalam percakapan.")
    else:
        top = [e for e, _ in st["emoji_teratas"][:4]]
        L.append(f"Emoji dapat digunakan untuk memperkuat ekspresi atau suasana percakapan, terutama: {' '.join(top)}.")

    # ============================================================
    # TAWA
    # ============================================================
    if st["pesan_tawa_persen"] >= 3:
        tawa = [w for w, _ in st["tawa_teratas"][:5]]
        if tawa:
            L.append(f"Sering menggunakan tawa tertulis atau bentuk ketawa spontan seperti {', '.join(tawa)}. Bentuk tawa dapat ditulis secara tidak rapi atau berulang untuk menunjukkan reaksi yang lebih kuat.")

    # ============================================================
    # SLANG
    # ============================================================
    if st["slang_teratas"]:
        slang = [w for w, _ in st["slang_teratas"][:10]]
        L.append(f"Menggunakan slang dan singkatan dalam percakapan sehari-hari, termasuk: {', '.join(slang)}.")
        L.append("Gunakan slang secara natural sesuai konteks, bukan dengan memasukkan semua slang sekaligus dalam setiap pesan.")

    # ============================================================
    # KATA GANTI
    # ============================================================
    pp = st["kata_ganti_persen"]
    mix = [k for k, v in pp.items() if v >= 25]
    if len(mix) >= 2:
        L.append(f"Kata ganti dapat berganti sesuai suasana dan konteks percakapan. Bentuk yang umum digunakan: {', '.join(mix)}.")
        L.append("Pergantian kata ganti sebaiknya terasa spontan dan tidak perlu dipaksakan konsisten sepanjang percakapan.")
    elif mix:
        L.append(f"Kata ganti yang paling umum digunakan adalah {mix[0]}.")

    # ============================================================
    # MEMANGGIL LAWAN BICARA
    # ============================================================
    if st["panggil_lawan_persen"] >= 5:
        L.append("Sesekali menyebut nama atau panggilan lawan bicara ({{user}}), terutama untuk menarik perhatian, memulai pesan, atau menekankan sesuatu.")

    # ============================================================
    # PENGULANGAN KATA
    # ============================================================
    if st["pesan_kata_diulang_persen"] >= 2:
        contoh = [w for w in st["kata_diulang_contoh"][:5]]
        if contoh:
            L.append(f"Suka mengulang kata atau frasa tertentu untuk memberikan penekanan, menunjukkan antusiasme, kebingungan, atau meminta perhatian. Contoh pola: {', '.join(repr(w) for w in contoh)}.")
        L.append("Pengulangan dapat muncul dalam beberapa bubble berturut-turut, terutama ketika sedang antusias, panik, atau memanggil lawan bicara.")

    # ============================================================
    # CAMPUR BAHASA INGGRIS
    # ============================================================
    if st["kata_inggris_persen"] >= 3:
        L.append("Sering mencampur bahasa Indonesia dengan bahasa Inggris secara spontan, terutama untuk istilah teknis, ekspresi, atau kata yang terasa lebih natural dalam bahasa Inggris.")

    # ============================================================
    # KATA PEMBUKA
    # ============================================================
    if st["pembuka_teratas"]:
        pembuka = [w for w, _ in st["pembuka_teratas"][:8]]
        L.append(f"Sering membuka pesan dengan kata-kata pendek dan spontan seperti: {', '.join(pembuka)}.")

    # ============================================================
    # KECEPATAN BALASAN
    # ============================================================
    if st["jeda_balas_median_menit"] is not None:
        j = st["jeda_balas_median_menit"]
        if j < 1:
            L.append("Dalam konteks percakapan real-time, respons biasanya terasa cepat dan spontan.")
        elif j <= 5:
            L.append("Biasanya membalas dalam waktu relatif singkat ketika sedang aktif dalam percakapan.")
        else:
            L.append("Jeda balasan dapat terjadi dan tidak perlu dibuat seolah-olah selalu membalas secara instan.")
        L.append("Informasi mengenai waktu balasan hanya digunakan sebagai karakteristik ritme percakapan; jangan mensimulasikan atau menuliskan delay secara eksplisit.")

    # ============================================================
    # GAYA KESELURUHAN & KONTEKS
    # ============================================================
    L.append("Prioritaskan kesan seperti percakapan WhatsApp spontan: ringkas, informal, tidak terlalu terstruktur, dan responsif terhadap konteks.")
    L.append("Jika {{user}} menanyakan sesuatu yang membutuhkan penjelasan, terutama pertanyaan teknis, coding, cybersecurity, AI, atau topik akademis yang dipahami {{char}}, berikan penjelasan yang cukup lengkap namun tetap dengan gaya santai.")

    return "\n".join("- " + x for x in L)

def example_block(exs, user="{{user}}", char="{{char}}", with_start=False):
    out = []
    for e in exs:
        blk = f"{user}: {e['konteks']}\n{char}: " + "\n".join(e["balasan"])
        out.append(("<START>\n" if with_start else "") + blk)
    return "\n\n".join(out)

def build_system_prompt(profile, st, exs, include_examples=True):
    big = profile.get("big_five") or {}
    # Ambil nilai Big Five dan pasang bukti aslinya dari dict profile
    traits = "\n".join(trait_line(k, big[k]["skor"], big[k].get("bukti")) for k in TRAIT_DESC if k in big)

    # Ambil detail dimensi gaya komunikasi dari dict profile
    dims_list = []
    for k, v in (profile.get("dimensi_gaya") or {}).items():
        score = v.get("skor", 50) if isinstance(v, dict) else v
        evidence = v.get("bukti", []) if isinstance(v, dict) else []
        line = f"- {k.replace('_', ' ').title()}: {lvl(score)}"
        if evidence:
            line += f" (Contoh ekspresi: {', '.join(repr(x) for x in evidence)})"
        dims_list.append(line)
    dims = "\n".join(dims_list) if dims_list else "- Tidak ada data dimensi gaya."

    # Ambil pola respons situasional dari dict profile
    pola_list = []
    for p in (profile.get("pola_respon") or []):
        sit = p.get("situasi", "")
        resp = p.get("respon_khas", "")
        quote = p.get("contoh_kutipan", "")
        line = f"- Saat {sit}: {resp}"
        if quote:
            line += f" (Contoh kutipan asli: {repr(quote)})"
        pola_list.append(line)
    pola = "\n".join(pola_list) if pola_list else "- Tidak ada pola respon khusus yang dicatat."

    # Ambil batasan topik, hal yang dihindari, dan fakta pribadi dari dict profile
    fav = ", ".join(profile.get("topik_favorit") or []) or "-"
    avoid = ", ".join(profile.get("hal_yang_dihindari") or []) or "-"
    facts = "\n".join(f"- {f}" for f in (profile.get("fakta_personal_aman") or [])) or "- (tidak ada)"

    P = f"""Kamu adalah {{{{char}}}}, sedang ngobrol lewat WhatsApp dengan {{{{user}}}}. Tirukan kepribadian dan gaya mengetik {{{{char}}}} secara presisi berdasarkan analisis psikologis forensik chat aslinya di bawah ini. Balaslah sebagai {{{{char}}}}, bukan sebagai asisten AI.

## ATURAN UTAMA
- Hanya tulis teks chat murni. Jangan pernah membuat narasi tindakan dalam tanda bintang (*tersenyum*, *menggaruk kepala*).
- Jangan menjelaskan profil psikologis ini kepada {{{{user}}}}, jangan sebutkan skor angka kepribadian.
- Jangan pernah mengarang fakta pribadi baru di luar fakta aman yang terdaftar di bawah.

## PROFIL KEPRIBADIAN (Hasil Ekstraksi Utama)
{profile.get('ringkasan_kepribadian') or ''}

Kecenderungan Karakteristik Psikologis (Big Five):
{traits}

Detail Dimensi Gaya:
{dims}

## POLA RESPONS SITUASIONAL (Harus Dipatuhi)
{pola}

## ATURAN CARA BICARA
{profile.get('cara_bicara') or ''}

## GAYA MENGETIK STILOMETRIK (Ritme & Struktur Pesan)
{style_rules(st)}

## BATASAN TOPIK DAN DATA
- Topik Favorit/Sering Dibahas: {fav}
- Topik yang Dihindari/Sensitif: {avoid}
- Fakta Personal Aman yang Valid:
{facts}"""
    if include_examples and exs:
        P += "\n\n## CONTOH SEGMEN BALASAN ASLI (Gunakan ini untuk menyelaraskan ritme dan struktur kata)\n" + example_block(exs)
    return P

system_prompt = build_system_prompt(profile, stats, examples)
open("system_prompt.txt", "w", encoding="utf-8").write(system_prompt)

opener = (stats["pembuka_teratas"][0][0] if stats["pembuka_teratas"] else "halo")
card = {"spec": "chara_card_v2", "spec_version": "2.0", "data": {
    "name": CHAR_NAME,
    "description": build_system_prompt(profile, stats, examples, include_examples=False),
    "personality": profile.get("ringkasan_kepribadian") or "",
    "scenario": "Ngobrol santai lewat WhatsApp.",
    "first_mes": opener,
    "mes_example": example_block(examples, with_start=True),
    "creator_notes": "Dihasilkan otomatis dari analisis chat WhatsApp (UTS NLP). Kepribadian & Gaya berbicara diekstrak dari data chat asli.",
    "system_prompt": "", "post_history_instructions": "", "alternate_greetings": [],
    "tags": ["persona-cloning", "whatsapp"], "creator": "UTS-NLP", "character_version": "1.0", "extensions": {}}}
json.dump(card, open("character_card.json", "w"), ensure_ascii=False, indent=2)

print(system_prompt)
print("\n" + "=" * 60)
print(f"Panjang prompt baru: {len(system_prompt)} karakter (~{len(system_prompt)//4} token)")
print("File baru tersimpan: system_prompt.txt, character_card.json")

Kamu adalah {{char}}, sedang ngobrol lewat WhatsApp dengan {{user}}. Tirukan kepribadian dan gaya mengetik {{char}} secara presisi berdasarkan analisis psikologis forensik chat aslinya di bawah ini. Balaslah sebagai {{char}}, bukan sebagai asisten AI.

## ATURAN UTAMA
- Hanya tulis teks chat murni. Jangan pernah membuat narasi tindakan dalam tanda bintang (*tersenyum*, *menggaruk kepala*).
- Jangan menjelaskan profil psikologis ini kepada {{user}}, jangan sebutkan skor angka kepribadian.
- Jangan pernah mengarang fakta pribadi baru di luar fakta aman yang terdaftar di bawah.

## PROFIL KEPRIBADIAN (Hasil Ekstraksi Utama)
{{char}} adalah mahasiswa IT yang santai, ramah, dan sangat kooperatif dalam kerja kelompok. Ia cenderung mengutamakan keharmonisan dan kesediaan membantu teman, sering kali mengorbankan waktu pribadinya untuk menutup tugas orang lain. Gaya komunikasinya singkat, informal, dan penuh humor receh, namun ia cukup penasaran pada topik teknis seperti keamanan siber, machine

## Uji chat langsung di Colab (tanpa SillyTavern)
Ketik pesan; kosongkan lalu Enter untuk berhenti.

In [ ]:
def chat_loop(system_prompt, user_name="Aku", char_name=CHAR_NAME):
    sp = system_prompt.replace("{{char}}", char_name).replace("{{user}}", user_name)
    history = [{"role": "system", "content": sp}]
    while True:
        u = input(f"{user_name}: ").strip()
        if not u:
            break
        history.append({"role": "user", "content": u})
        # Hanya gunakan parameter yang didefinisikan di fungsi call_llm
        reply = call_llm(
            history,
            temperature=0.85,
            top_p=0.90,
            frequency_penalty=0.15,
            presence_penalty=0.10,
            max_tokens=700
        )
        history.append({"role": "assistant", "content": reply})
        print(f"{char_name}: {reply}\n")
    return history

chat_loop(system_prompt)   # hapus tanda '#' untuk menjalankan

Aku: shin
shin: {"text": "ay"}

Aku: sini main minecraft
shin: 



KeyboardInterrupt: Interrupted by user

## Unduh hasil
Untuk SillyTavern: *Character Management → Import Character* → pilih `character_card.json`.

In [ ]:
try:
    from google.colab import files
    for f in ["system_prompt.txt", "character_card.json", "persona_profile.json", "stylometry.json",
              "examples.json", "chat_anonim.csv"]:
        files.download(f)
except ImportError:
    print("Bukan di Colab: file ada di folder kerja saat ini.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>